# 2. Baselines

This notebook runs the TF-IDF baseline and the BPE TF-IDF baseline.

In [ ]:
!python -m pip install -q numpy scikit-learn tokenizers

In [ ]:
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/final project/dataset')
else:
    PROJECT_DIR = Path.cwd()

print(f'Project directory: {PROJECT_DIR}')

In [ ]:
import tempfile
import warnings
from collections.abc import Callable, Iterable, Iterator, Sequence

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [ ]:
DATASET_NAME = 'tor_100w_2500tr.npz'
N_SITES = 100
N_TRACES = 100
BPE_VOCAB_SIZE = 1000

DATASET_PATH = PROJECT_DIR / DATASET_NAME

In [ ]:
def resolve_dataset_path(data_path: str | Path) -> Path:
    path = Path(data_path)

    if path.is_dir():
        subdirs = sorted(
            entry
            for entry in path.iterdir()
            if entry.is_dir() and not entry.name.startswith('__')
        )
        npz_files = sorted(
            entry
            for entry in path.iterdir()
            if entry.is_file() and entry.suffix.lower() == '.npz'
        )

        if subdirs:
            return path
        if npz_files:
            return npz_files[0]
        raise ValueError(f'No dataset directories or .npz files found in {path}')

    if path.is_file() and path.suffix.lower() == '.npz':
        return path

    raise ValueError(f'Dataset path does not exist or is unsupported: {path}')


def load_traces(
    data_path: str | Path,
    n_sites: int = 100,
    n_traces: int = 100,
) -> tuple[list[list[int]], list[int]]:
    resolved_path = resolve_dataset_path(data_path)

    with warnings.catch_warnings():
        warnings.filterwarnings(
            'ignore',
            category=np.exceptions.VisibleDeprecationWarning,
        )
        with np.load(resolved_path, allow_pickle=True) as dataset:
            data = dataset['data']
            raw_labels = np.asarray(dataset['labels'])

    unique_sites = np.unique(raw_labels)
    selected_sites = unique_sites[:n_sites]
    traces: list[list[int]] = []
    labels: list[int] = []
    site_to_id = {site: index for index, site in enumerate(selected_sites)}

    for site in selected_sites:
        site_indices = np.where(raw_labels == site)[0][:n_traces]
        for index in site_indices:
            traces.append(data[index].tolist())
            labels.append(site_to_id[site])

    return traces, labels


def burst_direction(value: int) -> str:
    return 'OUT' if value == 1 else 'IN'


def iter_bursts(trace: Sequence[int]) -> Iterator[tuple[str, int]]:
    index = 0
    while index < len(trace):
        current_value = trace[index]
        length = 1
        while index + length < len(trace) and trace[index + length] == current_value:
            length += 1
        yield burst_direction(current_value), length
        index += length


def burst_length_bucket(length: int) -> str:
    if length <= 5:
        return 'XS'
    if length <= 20:
        return 'S'
    if length <= 100:
        return 'M'
    return 'L'


def burst_tokenize(trace: Sequence[int]) -> list[str]:
    return [
        f'{direction}_{burst_length_bucket(length)}'
        for direction, length in iter_bursts(trace)
    ]


def proposal_length_bucket(length: int) -> str:
    if length <= 16:
        return f'LEN_{length:03d}'
    if length <= 32:
        start = ((length - 17) // 2) * 2 + 17
        end = start + 1
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 64:
        start = ((length - 33) // 4) * 4 + 33
        end = start + 3
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 128:
        start = ((length - 65) // 8) * 8 + 65
        end = start + 7
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 256:
        start = ((length - 129) // 16) * 16 + 129
        end = start + 15
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 512:
        start = ((length - 257) // 32) * 32 + 257
        end = start + 31
        return f'LEN_{start:03d}_{end:03d}'
    if length <= 1024:
        start = ((length - 513) // 64) * 64 + 513
        end = start + 63
        return f'LEN_{start:03d}_{end:03d}'
    return 'LEN_1025_PLUS'


def proposal_burst_tokenize(trace: Sequence[int]) -> list[str]:
    return [
        f'{direction}_{proposal_length_bucket(length)}'
        for direction, length in iter_bursts(trace)
    ]


def traces_to_documents(
    traces: Iterable[Sequence[int]],
    tokenize_fn: Callable[[Sequence[int]], list[str]],
) -> list[str]:
    return [' '.join(tokenize_fn(trace)) for trace in traces]


def train_bpe_tokenizer(documents: list[str], vocab_size: int = 1000) -> Tokenizer:
    tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=['[UNK]', '[CLS]', '[PAD]'])

    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as handle:
        corpus_path = Path(handle.name)
        for document in documents:
            handle.write(document + '\n')

    try:
        tokenizer.train(files=[str(corpus_path)], trainer=trainer)
    finally:
        corpus_path.unlink(missing_ok=True)

    return tokenizer

In [ ]:
traces, labels = load_traces(DATASET_PATH, n_sites=N_SITES, n_traces=N_TRACES)
print(f'Loaded {len(traces)} traces across {len(set(labels))} sites')

In [ ]:
tfidf_docs = traces_to_documents(traces, tokenize_fn=burst_tokenize)
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_features = tfidf_vectorizer.fit_transform(tfidf_docs)
tfidf_classifier = LogisticRegression(max_iter=1000, C=1.0)
tfidf_scores = cross_val_score(tfidf_classifier, tfidf_features, labels, cv=5)

print(f'Document-term matrix shape: {tfidf_features.shape}')
print(f'TF-IDF baseline accuracy: {tfidf_scores.mean():.3f} (+/- {tfidf_scores.std():.3f})')

In [ ]:
proposal_docs = traces_to_documents(traces, tokenize_fn=proposal_burst_tokenize)
bpe_tokenizer = train_bpe_tokenizer(proposal_docs, vocab_size=BPE_VOCAB_SIZE)
bpe_docs = [' '.join(bpe_tokenizer.encode(doc).tokens) for doc in proposal_docs]
bpe_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
bpe_features = bpe_vectorizer.fit_transform(bpe_docs)
bpe_classifier = LogisticRegression(max_iter=1000, C=1.0)
bpe_scores = cross_val_score(bpe_classifier, bpe_features, labels, cv=5)

print(f'BPE tokens (first 15): {bpe_tokenizer.encode(proposal_docs[0]).tokens[:15]}')
print(f'BPE vocab size: {bpe_tokenizer.get_vocab_size()}')
print(f'BPE TF-IDF accuracy: {bpe_scores.mean():.3f} (+/- {bpe_scores.std():.3f})')